##### Support Request Generator Stream

Create a scheduled job to generate support requests.

In [ ]:
%pip install --upgrade databricks-sdk

In [ ]:
from databricks.sdk import WorkspaceClient
import databricks.sdk.service.jobs as j
import os

w = WorkspaceClient()
notebook_abs_path = os.path.abspath("../jobs/support_request_generator")
notebook_dbx_path = notebook_abs_path.replace(
    os.environ.get("DATABRICKS_WORKSPACE_ROOT", "/Workspace"),
    "/Workspace"
)

job_name = "Support Request Generator Stream"
new_settings = j.JobSettings(
    name=job_name,
    schedule=j.CronSchedule(
        quartz_cron_expression="0 0/10 * * * ?",
        timezone_id="UTC",
        pause_status=j.PauseStatus.UNPAUSED,
    ),
    tasks=[
        j.Task(
            task_key="support_request_generator_stream",
            notebook_task=j.NotebookTask(
                notebook_path=notebook_dbx_path,
                base_parameters={
                    "CATALOG": dbutils.widgets.get("CATALOG"),
                    "SUPPORT_RATE": dbutils.widgets.get("SUPPORT_RATE"),
                    "LLM_MODEL": dbutils.widgets.get("LLM_MODEL"),
                },
            ),
        )
    ],
)

matches = sorted(
    [job for job in w.jobs.list(name=job_name) if job.settings and job.settings.name == job_name],
    key=lambda x: x.created_time or 0,
    reverse=True,
)
if matches:
    target_job_id = matches[0].job_id
    w.jobs.reset(job_id=target_job_id, new_settings=new_settings)
    job = w.jobs.get(job_id=target_job_id)
    print(f"Updated existing job_id={target_job_id}")

    for stale in matches[1:]:
        stale_id = stale.job_id
        for run in w.jobs.list_runs(job_id=stale_id, active_only=True):
            w.jobs.cancel_run(run_id=run.run_id)
        w.jobs.delete(job_id=stale_id)
        print(f"Deleted stale duplicate job_id={stale_id}")
else:
    settings_payload = new_settings.as_dict() if hasattr(new_settings, "as_dict") else new_settings
    created = w.api_client.do("POST", "/api/2.2/jobs/create", body=settings_payload)
    target_job_id = created["job_id"] if isinstance(created, dict) else created.job_id
    job = w.jobs.get(job_id=target_job_id)
    print(f"Created job_id={target_job_id}")

import sys
sys.path.append('../utils')
from uc_state import add

add(dbutils.widgets.get("CATALOG"), "jobs", job)
w.jobs.run_now(job_id=target_job_id)